# Cvičenie 8

Šum a jeho odstránenie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import skimage as sk

## Úloha 1

Porovnanie druhov šumu - pomocou funkcii skimage vygenerujte zašumné obrázky.

In [ ]:
image = sk.util.img_as_float(sk.data.camera())
rng = np.random.default_rng(seed=6)
sigma = 0.1

noisy_gauss = ... # s tymto obrazkom budeme pracovat pri odsumovani
impulse = ...
peak = 30      # menší peak = silnejší šum
poisson = rng.poisson(image * peak) / peak  # >= 0, ale môže presiahnuť 1

speckle = ...

fig, ax = plt.subplots(1, 5, figsize=(17, 5))
for a, obr, nazov in zip(ax, [image, noisy_gauss, impulse, poisson, speckle],
                         ["Original", "Gauss", "Impulzny", "Poissonov", "Multiplikativny"]):
    a.imshow(np.clip(obr, 0, 1), cmap="gray", vmin=0, vmax=1)
    a.set_title(f"{nazov} PSNR:{sk.metrics.peak_signal_noise_ratio(image, obr):.2f}" )
    a.axis("off")
plt.tight_layout()
plt.show()

## Úloha 2

Odhad úrovne šumu - doplňte výrez homogénnej oblasti obrázka a porovnajte s "estimate_sigma" zo skimage.

In [ ]:
plocha = noisy_gauss[...]
plt.imshow(plocha, cmap="gray")
sk_odhad = ...
print(f"jednoduchy odhad: {plocha.std():.3f} \nsk funkcia: {sk_odhad:.3f} \nskutočná hodnota: {sigma}")

## Úloha 3

Fúzia obrázkov - doplňte funkciu fuzia (napríklad podľa prednášky).

In [ ]:
def fuzia(pocet=5, sigma=0.1):
    skupina = ... # TODO
    priemer = ... # TODO
    psnr_1 = sk.metrics.peak_signal_noise_ratio(image, skupina[..., 0], data_range=1)
    psnr_N = sk.metrics.peak_signal_noise_ratio(image, priemer, data_range=1)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(skupina[..., 0], cmap="gray", vmin=0, vmax=1)
    ax[0].set_title(f"1 zašumený obraz, PSNR = {psnr_1:.1f} dB")
    ax[1].imshow(priemer, cmap="gray", vmin=0, vmax=1)
    ax[1].set_title(f"priemer z {pocet}: PSNR = {psnr_N:.1f} dB )")
    for a in ax:
        a.axis("off")
    plt.show()

widgets.interact(
    fuzia,
    pocet=widgets.IntSlider(value=5, min=1, max=100, description="počet",
                            continuous_update=False),
    sigma=widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description="σ",
                              continuous_update=False),
);

## Úloha 4

Odšumovanie - Denoising
Vyberte 4 metódy - 2 klasické (z prednášky) a 2 neurónové (z dokumentácie deep inverse).
Pomocou šablóny na konci spravte ich porovnanie.

**Vybraté metódy by ste mali vedieť stručne opísať.**

In [ ]:
import torch
import deepinv as dinv

device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")

def na_tenzor(obraz):            # numpy (H, W) → tenzor (1, 1, H, W)
    return torch.from_numpy(obraz.astype(np.float32))[None, None].to(device)

def na_numpy(t):                 # tenzor (1, 1, H, W) → numpy (H, W)
    return t.detach().cpu().numpy()[0, 0]

In [ ]:
denoised_1 = ...

In [ ]:
denoised_2 = ...

In [ ]:
denoised_3 = ...

In [ ]:
denoised_4 = ...

In [ ]:
vysledky = {
    "Nazov 1":  denoised_1,
    "Nazov 2":  denoised_2,
    "Nazov 3": denoised_3,
    "Nazov 4": denoised_4,
}

psnr = [sk.metrics.peak_signal_noise_ratio(image, v, data_range=1) for v in vysledky.values()]

In [ ]:
dinv.utils.plot(
    plot_inset=True,
    img_list=[na_tenzor(image), na_tenzor(noisy_gauss)] + [na_tenzor(v) for v in vysledky.values()],
    titles=["Originál", "Zašumený"] + list(vysledky),
    subtitles=["PSNR:", f"{sk.metrics.peak_signal_noise_ratio(image, noisy_gauss, data_range=1):.2f} dB"]
              + [f"{p:.2f} dB" for p in psnr],
    extract_size=0.2, extract_loc=(0.4, 0.2), inset_size=0.5, figsize=(17, 5),
)